In [ ]:
import numpy as np
import pandas as pd
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, classification_report
import os

###############################################################################
# 1) 데이터 로드 및 Test Set 분리 (난수 42 고정)
###############################################################################
df = pd.read_pickle("/Users/gyuminkang/Desktop/sci/LSWMD.pkl")

def safe_extract_label(val):
    try:
        if isinstance(val, str): return val.strip()
        if len(val) > 0 and len(val[0]) > 0: return str(val[0][0]).strip()
        return 'none'
    except:
        return 'none'

df['failureType_str'] = df['failureType'].apply(safe_extract_label)
df = df[df['failureType_str'] != 'none'].copy()

def resize_image(image, target_size=(64, 64)):
    if np.max(image) == 2:
        image = np.where(image == 2, 1, 0).astype(np.uint8)
    else:
        image = image.astype(np.uint8)
    return cv2.resize(image, target_size, interpolation=cv2.INTER_NEAREST)

print("데이터 전처리 중...")
processed_images = [resize_image(image) for image in df['waferMap']]
labels = df['failureType_str'].tolist()

le = LabelEncoder()
encoded_labels = le.fit_transform(labels)
class_names = list(le.classes_)

# 🚨 완벽히 격리된 20% 테스트 셋 구성
_, X_test, _, y_test = train_test_split(
    processed_images, encoded_labels, test_size=0.2, random_state=42, stratify=encoded_labels
)

X_test = np.array(X_test, dtype=np.float32)
y_test = np.array(y_test, dtype=np.int64)

###############################################################################
# 2) 모델 아키텍처 정의
###############################################################################
class Shortcut3ResNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(Shortcut3ResNetBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False), nn.BatchNorm2d(out_channels))
        else:
            self.shortcut = nn.Sequential()
    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(identity)
        return F.relu(out)

class CNN5(nn.Module):
    def __init__(self, num_blocks):
        super(CNN5, self).__init__()
        self.layers = nn.ModuleList()
        in_channels, out_channels = 1, 32
        for _ in range(num_blocks):
            self.layers.append(Shortcut3ResNetBlock(in_channels, out_channels))
            self.layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            in_channels = out_channels
            out_channels *= 2
        self.extractor = nn.Sequential(*self.layers)
        self.final_conv = nn.Conv2d(out_channels // 2, 1024, kernel_size=1)
    def forward(self, x):
        x = self.extractor(x)
        x = self.final_conv(x)
        x = F.adaptive_avg_pool2d(x, (1, 1))
        return x.view(x.size(0), -1)

class WindowedLSTM(nn.Module):
    def __init__(self, input_size=1024, hidden_size=1024, num_windows=16):
        super(WindowedLSTM, self).__init__()
        self.num_windows = num_windows
        self.window_length = input_size // num_windows
        self.lstm = nn.LSTM(input_size=self.window_length, hidden_size=hidden_size, batch_first=True, bidirectional=True)
        self.Wk = nn.Linear(2 * hidden_size, 2 * hidden_size)
        self.Wq = nn.Linear(2 * hidden_size, 2 * hidden_size)
        self.v  = nn.Linear(2 * hidden_size, 1)
    def forward(self, x):
        x = x.view(x.size(0), self.num_windows, self.window_length)
        H, _ = self.lstm(x)  
        query = H[:, -1, :].unsqueeze(1)
        keys = H  
        linear_sum = self.Wk(keys) + self.Wq(query)
        mish_out = linear_sum * torch.tanh(F.softplus(linear_sum))
        energy = self.v(mish_out)  
        attention_weights = F.softmax(energy, dim=1)  
        return torch.sum(attention_weights * keys, dim=1)

class CombinedModel(nn.Module):
    def __init__(self, cnn, lstm, num_classes):
        super(CombinedModel, self).__init__()
        self.cnn = cnn
        self.lstm = lstm
        self.fc = nn.Linear(1024 + 1024 * 2, num_classes) 
    def forward(self, x):
        return self.fc(torch.cat((self.cnn(x), self.lstm(self.cnn(x))), dim=1))

device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')

###############################################################################
# 3) 진짜 앙상블(True Ensemble) 평가 함수
###############################################################################
def evaluate_ensemble(models, dataloader, class_names):
    for m in models:
        m.eval()
        
    all_preds = []
    all_labels = []

    print(f"\n🚀 {len(models)}개의 모델이 '집단 지성'으로 테스트 데이터를 추론 중입니다...")
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            # 모델 개수만큼의 예측 확률을 더할 빈 텐서 준비
            ensemble_probs = torch.zeros(inputs.size(0), len(class_names)).to(device)
            
            # 각 모델이 예측한 확률(Softmax)을 누적 합산 (Soft Voting)
            for m in models:
                outputs = m(inputs)
                probs = F.softmax(outputs, dim=1)
                ensemble_probs += probs
                
            # 전체 모델의 평균 확률로 최종 결론 도출
            ensemble_probs = ensemble_probs / len(models)
            _, predicted = torch.max(ensemble_probs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True, zero_division=0)
    overall_accuracy = 100.0 * (all_preds == all_labels).sum() / len(all_labels)
    macro_precision = report_dict["macro avg"]["precision"]
    macro_recall    = report_dict["macro avg"]["recall"]
    macro_f1        = report_dict["macro avg"]["f1-score"]

    cm = confusion_matrix(all_labels, all_preds)
    row_sums = cm.sum(axis=1)
    class_acc = np.divide(cm.diagonal(), row_sums, out=np.zeros_like(cm.diagonal(), dtype=float), where=row_sums!=0)

    print("\n===================================================================")
    print(f"       🏆 SCRBLAA-Net {len(models)}-Model 집단 지성 앙상블 평가 리포트 🏆       ")
    print("===================================================================")
    for i, acc in enumerate(class_acc):
        print(f" - {class_names[i]:<12}: {acc * 100:>6.2f}%")
    print("-" * 50)
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4, zero_division=0))
    print("-" * 50)
    print(f" 🎯 앙상블 전체 정확도 (Accuracy) : {overall_accuracy:.2f}%")
    print(f" 🚀 앙상블 매크로 F1-Score        : {macro_f1:.4f}")
    print("===================================================================\n")

###############################################################################
# 4) 메인 실행부
###############################################################################
if __name__ == "__main__":
    
    BASE_DIR = "/Users/gyuminkang/Desktop/sci/"
    models = []
    
    # 🚨 1. 5개의 K-Fold 가중치 파일 불러오기
    for i in range(1, 6):
        path = os.path.join(BASE_DIR, f"scrblaa_net_fold_{i}_best.pth")
        
        if not os.path.exists(path):
            print(f"❌ 오류: {path} 파일을 찾을 수 없습니다. 5개의 폴드 파일이 모두 있어야 합니다.")
            exit()
            
        print(f"📂 폴드 모델 {i} 뇌 이식 중... ({path})")
        m = CombinedModel(CNN5(num_blocks=5), WindowedLSTM(input_size=1024, hidden_size=1024, num_windows=16), num_classes=len(class_names)).to(device)
        m.load_state_dict(torch.load(path, map_location=device, weights_only=False))
        models.append(m)
        
    # 🚨 2. 최종 전체 학습(Full Retraining) 마스터 가중치 파일 추가로 불러오기
    final_path = os.path.join(BASE_DIR, "scrblaa_net_final_weights.pth")
    if os.path.exists(final_path):
        print(f"🔥 최종 마스터 모델 뇌 이식 중... ({final_path})")
        final_m = CombinedModel(CNN5(num_blocks=5), WindowedLSTM(input_size=1024, hidden_size=1024, num_windows=16), num_classes=len(class_names)).to(device)
        final_m.load_state_dict(torch.load(final_path, map_location=device, weights_only=False))
        models.append(final_m)
        print(f"✅ 총 {len(models)}개의 모델(5-Fold + Final)이 완벽하게 결성되었습니다!")
    else:
        print(f"⚠️ 경고: 최종 가중치 파일({final_path})을 찾을 수 없습니다. 기존 5개 모델로만 진행합니다.")

    # 테스트 데이터 VRAM 초고속 적재
    X_test_t = torch.from_numpy(X_test).unsqueeze(1).to(device)
    y_test_t = torch.from_numpy(y_test).to(device)
    test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64, shuffle=False)

    # 앙상블 평가 시작!
    evaluate_ensemble(models, test_loader, class_names)

In [ ]:
import numpy as np
import pandas as pd
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, classification_report
import os

###############################################################################
# 1) 데이터 로드 및 Test Set 분리 (난수 42 고정)
###############################################################################
df = pd.read_pickle("/Users/gyuminkang/Desktop/sci/LSWMD.pkl")

def safe_extract_label(val):
    try:
        if isinstance(val, str): return val.strip()
        if len(val) > 0 and len(val[0]) > 0: return str(val[0][0]).strip()
        return 'none'
    except:
        return 'none'

df['failureType_str'] = df['failureType'].apply(safe_extract_label)
df = df[df['failureType_str'] != 'none'].copy()

def resize_image(image, target_size=(64, 64)):
    if np.max(image) == 2:
        image = np.where(image == 2, 1, 0).astype(np.uint8)
    else:
        image = image.astype(np.uint8)
    return cv2.resize(image, target_size, interpolation=cv2.INTER_NEAREST)

print("데이터 전처리 중...")
processed_images = [resize_image(image) for image in df['waferMap']]
labels = df['failureType_str'].tolist()

le = LabelEncoder()
encoded_labels = le.fit_transform(labels)
class_names = list(le.classes_)

# 🚨 완벽히 격리된 20% 테스트 셋 구성
_, X_test, _, y_test = train_test_split(
    processed_images, encoded_labels, test_size=0.2, random_state=42, stratify=encoded_labels
)

X_test = np.array(X_test, dtype=np.float32)
y_test = np.array(y_test, dtype=np.int64)

###############################################################################
# 2) 모델 아키텍처 정의
###############################################################################
class Shortcut3ResNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(Shortcut3ResNetBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False), nn.BatchNorm2d(out_channels))
        else:
            self.shortcut = nn.Sequential()
    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(identity)
        return F.relu(out)

class CNN5(nn.Module):
    def __init__(self, num_blocks):
        super(CNN5, self).__init__()
        self.layers = nn.ModuleList()
        in_channels, out_channels = 1, 32
        for _ in range(num_blocks):
            self.layers.append(Shortcut3ResNetBlock(in_channels, out_channels))
            self.layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            in_channels = out_channels
            out_channels *= 2
        self.extractor = nn.Sequential(*self.layers)
        self.final_conv = nn.Conv2d(out_channels // 2, 1024, kernel_size=1)
    def forward(self, x):
        x = self.extractor(x)
        x = self.final_conv(x)
        x = F.adaptive_avg_pool2d(x, (1, 1))
        return x.view(x.size(0), -1)

class WindowedLSTM(nn.Module):
    def __init__(self, input_size=1024, hidden_size=1024, num_windows=16):
        super(WindowedLSTM, self).__init__()
        self.num_windows = num_windows
        self.window_length = input_size // num_windows
        self.lstm = nn.LSTM(input_size=self.window_length, hidden_size=hidden_size, batch_first=True, bidirectional=True)
        self.Wk = nn.Linear(2 * hidden_size, 2 * hidden_size)
        self.Wq = nn.Linear(2 * hidden_size, 2 * hidden_size)
        self.v  = nn.Linear(2 * hidden_size, 1)
    def forward(self, x):
        x = x.view(x.size(0), self.num_windows, self.window_length)
        H, _ = self.lstm(x)  
        query = H[:, -1, :].unsqueeze(1)
        keys = H  
        linear_sum = self.Wk(keys) + self.Wq(query)
        mish_out = linear_sum * torch.tanh(F.softplus(linear_sum))
        energy = self.v(mish_out)  
        attention_weights = F.softmax(energy, dim=1)  
        return torch.sum(attention_weights * keys, dim=1)

class CombinedModel(nn.Module):
    def __init__(self, cnn, lstm, num_classes):
        super(CombinedModel, self).__init__()
        self.cnn = cnn
        self.lstm = lstm
        self.fc = nn.Linear(1024 + 1024 * 2, num_classes) 
    def forward(self, x):
        return self.fc(torch.cat((self.cnn(x), self.lstm(self.cnn(x))), dim=1))

device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')

###############################################################################
# 3) 진짜 앙상블(True Ensemble) 평가 함수
###############################################################################
def evaluate_ensemble(models, dataloader, class_names):
    for m in models:
        m.eval()
        
    all_preds = []
    all_labels = []

    print(f"\n🚀 {len(models)}개의 모델이 '집단 지성'으로 테스트 데이터를 추론 중입니다...")
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            # 모델 개수만큼의 예측 확률을 더할 빈 텐서 준비
            ensemble_probs = torch.zeros(inputs.size(0), len(class_names)).to(device)
            
            # 각 모델이 예측한 확률(Softmax)을 누적 합산 (Soft Voting)
            for m in models:
                outputs = m(inputs)
                probs = F.softmax(outputs, dim=1)
                ensemble_probs += probs
                
            # 전체 모델의 평균 확률로 최종 결론 도출
            ensemble_probs = ensemble_probs / len(models)
            _, predicted = torch.max(ensemble_probs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True, zero_division=0)
    overall_accuracy = 100.0 * (all_preds == all_labels).sum() / len(all_labels)
    macro_precision = report_dict["macro avg"]["precision"]
    macro_recall    = report_dict["macro avg"]["recall"]
    macro_f1        = report_dict["macro avg"]["f1-score"]

    cm = confusion_matrix(all_labels, all_preds)
    row_sums = cm.sum(axis=1)
    class_acc = np.divide(cm.diagonal(), row_sums, out=np.zeros_like(cm.diagonal(), dtype=float), where=row_sums!=0)

    print("\n===================================================================")
    print(f"       🏆 SCRBLAA-Net {len(models)}-Model 집단 지성 앙상블 평가 리포트 🏆       ")
    print("===================================================================")
    for i, acc in enumerate(class_acc):
        print(f" - {class_names[i]:<12}: {acc * 100:>6.2f}%")
    print("-" * 50)
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4, zero_division=0))
    print("-" * 50)
    print(f" 🎯 앙상블 전체 정확도 (Accuracy) : {overall_accuracy:.2f}%")
    print(f" 🚀 앙상블 매크로 F1-Score        : {macro_f1:.4f}")
    print("===================================================================\n")

###############################################################################
# 4) 메인 실행부
###############################################################################
if __name__ == "__main__":
    
    BASE_DIR = "/Users/gyuminkang/Desktop/sci/"
    models = []
    
    # 🚨 1. 5개의 K-Fold 가중치 파일 불러오기
    for i in range(1, 6):
        path = os.path.join(BASE_DIR, f"scrblaa_net_fold_{i}_best.pth")
        
        if not os.path.exists(path):
            print(f"❌ 오류: {path} 파일을 찾을 수 없습니다. 5개의 폴드 파일이 모두 있어야 합니다.")
            exit()
            
        print(f"📂 폴드 모델 {i} 뇌 이식 중... ({path})")
        m = CombinedModel(CNN5(num_blocks=5), WindowedLSTM(input_size=1024, hidden_size=1024, num_windows=16), num_classes=len(class_names)).to(device)
        m.load_state_dict(torch.load(path, map_location=device, weights_only=False))
        models.append(m)
        
    # 🚨 2. 최종 전체 학습(Full Retraining) 마스터 가중치 파일 추가로 불러오기
    final_path = os.path.join(BASE_DIR, "scrblaa_net_final_weights.pth")
    if os.path.exists(final_path):
        print(f"🔥 최종 마스터 모델 뇌 이식 중... ({final_path})")
        final_m = CombinedModel(CNN5(num_blocks=5), WindowedLSTM(input_size=1024, hidden_size=1024, num_windows=16), num_classes=len(class_names)).to(device)
        final_m.load_state_dict(torch.load(final_path, map_location=device, weights_only=False))
        models.append(final_m)
        print(f"✅ 총 {len(models)}개의 모델(5-Fold + Final)이 완벽하게 결성되었습니다!")
    else:
        print(f"⚠️ 경고: 최종 가중치 파일({final_path})을 찾을 수 없습니다. 기존 5개 모델로만 진행합니다.")

    # 테스트 데이터 VRAM 초고속 적재
    X_test_t = torch.from_numpy(X_test).unsqueeze(1).to(device)
    y_test_t = torch.from_numpy(y_test).to(device)
    test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64, shuffle=False)

    # 앙상블 평가 시작!
    evaluate_ensemble(models, test_loader, class_names)

데이터 전처리 중...
📂 폴드 모델 1 뇌 이식 중... (/Users/gyuminkang/Desktop/sci/scrblaa_net_fold_1_best.pth)
📂 폴드 모델 2 뇌 이식 중... (/Users/gyuminkang/Desktop/sci/scrblaa_net_fold_2_best.pth)
📂 폴드 모델 3 뇌 이식 중... (/Users/gyuminkang/Desktop/sci/scrblaa_net_fold_3_best.pth)
📂 폴드 모델 4 뇌 이식 중... (/Users/gyuminkang/Desktop/sci/scrblaa_net_fold_4_best.pth)
📂 폴드 모델 5 뇌 이식 중... (/Users/gyuminkang/Desktop/sci/scrblaa_net_fold_5_best.pth)
🔥 최종 마스터 모델 뇌 이식 중... (/Users/gyuminkang/Desktop/sci/scrblaa_net_final_weights.pth)
✅ 총 6개의 모델(5-Fold + Final)이 완벽하게 결성되었습니다!

🚀 6개의 모델이 '집단 지성'으로 테스트 데이터를 추론 중입니다...

       🏆 SCRBLAA-Net 6-Model 집단 지성 앙상블 평가 리포트 🏆       
 - Center      :  97.90%
 - Donut       :  87.39%
 - Edge-Loc    :  92.77%
 - Edge-Ring   :  98.71%
 - Loc         :  83.45%
 - Near-full   :  93.33%
 - Random      :  94.80%
 - Scratch     :  83.61%
--------------------------------------------------
              precision    recall  f1-score   support

      Center     0.9622    0.9790    0.9706       859
    